# VXN/VIX 科技相对隐含风险：来源入口

固定三个可投资桶 SOXL/SOXX/cash，不新增交易资产。此处只核查一个信息来源，不做投资建议或模型选择。来源证据与实际历史版本可得性独立报告。


In [1]:
import os, sys, json
from pathlib import Path
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "SPEC.md").exists())
sys.path.insert(0, str(root))
import qlib
from qlib.data import D
from qlib.workflow import R
card = root / "research" / "vxn_vix_source_qualification_20261011"
source = json.loads((card / "source_qualification.json").read_text())
provider = root / "data" / "qlib" / "published_futu_affine_20260925_v1"
qlib.init(provider_uri=str(provider), region="us", kernels=1)
calendar = D.calendar(start_time="2014-04-17", end_time="2014-12-31", freq="day")
upper_bound = len(calendar)
assert upper_bound < 252
print("Before 2015 inner cutoff: upper bound", upper_bound, "sessions; source could not meet 252-row minimum for 2016 outer.")


2026/10/11 16:07:09 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[67768:MainThread](2026-10-11 16:07:09,971) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[67768:MainThread](2026-10-11 16:07:09,972) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[67768:MainThread](2026-10-11 16:07:09,972) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


Before 2015 inner cutoff: upper bound 179 sessions; source could not meet 252-row minimum for 2016 outer.


In [2]:
from src.data.alfred_initial_release import read_initial_release_archive, qualify_paired_information, build_information_store
from src.data.provider_content_hash import provider_content_hash
import pandas as pd, numpy as np, hashlib
price_hash_before = provider_content_hash(provider)[0]
raw = root / "data" / "raw" / "external" / "alfred_vxn_vix_initial_20261011_v1"
vxn = read_initial_release_archive(raw / "VXNCLS", as_of="2026-09-25")
vix = read_initial_release_archive(raw / "VIXCLS", as_of="2026-09-25")
qlib.init(provider_uri=str(provider), region="us", kernels=1)
full_calendar = pd.DatetimeIndex(D.calendar(start_time="2011-01-03", end_time="2026-09-25", freq="day"))
state, paired, audit = qualify_paired_information(vxn, vix, full_calendar)
store = root / "data" / "qlib" / "alfred_tech_iv_initial_20261011_v1"
manifest, native = build_information_store(state, provider_uri=store, as_of="2026-09-25", source="official_alfred_initial_release_20261011_v1")
assert provider_content_hash(provider)[0] == price_hash_before
eligibility = []
for outer in range(2016, 2027):
    cutoff = pd.Timestamp(str(outer-1) + "-01-01", tz="America/New_York")
    mask = state.known.eq(1) & state.available_at.lt(cutoff)
    eligibility.append(dict(outer_year=outer, inner_year=outer-1, qualified_source_rows_before_inner_cut=int(mask.sum()), source_training_minimum_met=bool(mask.sum()>=252)))
eligibility = pd.DataFrame(eligibility)
assert not eligibility.loc[eligibility.outer_year.eq(2016), "source_training_minimum_met"].iloc[0]
with R.start(experiment_name=source["experiment"], recorder_name="initial_release_source_qualification", uri=str(card / "qlib_recorder")):
    rec = R.get_recorder()
    rec.log_params(models_trained=0, replays=0, initial_release_payload_available=True, source_store=str(store))
    rec.save_objects(**{"raw_vxn.pkl": vxn, "raw_vix.pkl": vix, "paired_source_events.pkl": paired, "source_state.pkl": state, "source_readback.pkl": native, "source_audit.pkl": audit})
summary = dict(experiment=source["experiment"], status="source_qualified_with_quarantine_research_not_authorized", models_trained=0, replays=0,
    promotion_eligible=False, initial_release_payload_available=True, source_audit=audit,
    recorder_id=rec.id, experiment_id=str(rec.experiment_id), recorder_uri=str(card / "qlib_recorder"), recorder_local_dir=rec.get_local_dir(), recorder_status=rec.status,
    provider_uri=str(store), price_provider_uri=str(provider), price_provider_unchanged=True, as_of="2026-09-25", region="us",
    artifact_policy=dict(policy="slim_card_recorder_first", workflow_config="workflow.yaml", recorder_only_artifacts=["raw_vxn.pkl","raw_vix.pkl","paired_source_events.pkl","source_state.pkl","source_readback.pkl","source_audit.pkl"]))
(card / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n")
eligibility.to_csv(card / "source_eligibility.csv", index=False)
paired.loc[paired.premature_release].assign(observation_date=lambda x:x.observation_date.astype(str)).to_csv(card / "source_anomalies.csv", index=False)
known = state.loc[state.known.eq(1)]
print(json.dumps(audit, indent=2, ensure_ascii=False))
display(eligibility)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[67768:MainThread](2026-10-11 16:07:10,794) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[67768:MainThread](2026-10-11 16:07:10,795) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[67768:MainThread](2026-10-11 16:07:10,796) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


[67768:MainThread](2026-10-11 16:07:11,625) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[67768:MainThread](2026-10-11 16:07:11,662) INFO - qlib.workflow - [exp.py:258] - Experiment 686259316940943595 starts running ...


[67768:MainThread](2026-10-11 16:07:11,795) INFO - qlib.workflow - [recorder.py:345] - Recorder 2ae4bc644d674f5bbc88cda0125ee1de starts running under Experiment 686259316940943595 ...


[67768:MainThread](2026-10-11 16:07:12,575) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{
  "raw_vxn_rows": 3243,
  "raw_vix_rows": 4072,
  "calendar_rows": 3956,
  "known_signal_rows": 3126,
  "unknown_signal_rows": 830,
  "premature_market_observations": 1,
  "non_session_raw_observations": 119,
  "unknown_latest_observations": 1,
  "date_level_initial_release_verified": true,
  "intraday_publication_timestamp_verified": false,
  "availability_contract": "next native session09:31NY after max(observation_date,both release dates); premature rows quarantined",
  "same_observation_date_pairing": true,
  "unknown_feature_not_zero_filled": true,
  "price_evaluation_dates_dropped": 0,
  "source_range": [
    "2011-01-03",
    "2026-09-25"
  ]
}


,outer_year,inner_year,qualified_source_rows_before_inner_cut,source_training_minimum_met
0,2016,2015,177,False
1,2017,2016,429,True
2,2018,2017,681,True
3,2019,2018,933,True
4,2020,2019,1182,True
5,2021,2020,1434,True
6,2022,2021,1687,True
7,2023,2022,1939,True
8,2024,2023,2190,True
9,2025,2024,2440,True


# VXN/VIX 首次发布来源资格

已实际取得匿名官方 Initial Release Only ZIP，并通过 Qlib 原生信息 Store 读回；未训练、未回测、不晋升。

原始 VXN3243/VIX4072条，无重复；完整3956个价格交易日保留，3126日有可用信息、830日未知。配对必须同观察日，公开日期保守映射到下一真实交易日09:31。2018-10-08真实市场日存在提前发布日期标记，已隔离，未编造原因或改写原记录。缺值不被零或旧值填补。

2016外层的内层2015截止前只有177个已知来源日（尚未扣除标签成熟尾部），不能满足252行；必须明确SOXX退回。后续比较的新信息模型与价格对照必须共享同一来源有效样本，不能将训练跨度差异归因于信息增量。

下一步预登记一个固定log(VXN/VIX)信息增量，保持费用/回撤/年度/滚动/置信区间风险门。来源合格不代表alpha，H20及已失败广义CBOE路线保持关闭。

官方来源：[ALFRED VXN](https://alfred.stlouisfed.org/series?seid=VXNCLS)、[首次发布格式](https://alfred.stlouisfed.org/help/downloaddata)、[CBOE 方法](https://cdn.cboe.com/api/global/us_indices/governance/Volatility_Index_Methodology_Selected_Broad_Based_Index_Equity_and_ETF_Volatility_Indices.pdf)。
